# Limpieza · Encuesta de eventos virtuales 2026

Entrada: `datos/originales/Eventos Virtuales ACRIP 2026_anonimizada.xlsx` (hoja `Responses`)

Salida: `datos/limpios/encuestas_virtuales.csv`

Llaves: `respuesta_id`, `persona_id`, `empresa_id`. Un registro por respuesta.

## 1. Librerías y rutas

In [ ]:
import pandas as pd

from utils import ORIGINALES, LIMPIOS, PATRON_EMPRESA, PATRON_PERSONA, fecha_es, ciudad_estandar, actualizaciones_estandar

## 2. Cargar la hoja `Responses`

La fila 1 es un título y la 2 son los encabezados. Se renombran las 19 columnas por posición para trabajar con nombres cortos.

In [ ]:
ARCHIVO = "Eventos Virtuales ACRIP 2026_anonimizada.xlsx"

crudo = pd.read_excel(ORIGINALES / ARCHIVO, sheet_name="Responses", header=1)
crudo = crudo.dropna(subset=[crudo.columns[1]]).reset_index(drop=True)

NOMBRES = ["n", "respuesta_id", "inicio", "fin", "ip", "recopilador", "nota_espacio", "nota_convocatoria",
           "nota_conferencistas", "nota_temas", "temas_interes", "acepta_actualizaciones", "persona_id",
           "empresa_id", "ciudad", "telefono", "correo", "empresa_en_crm", "metodo_asignacion"]
assert len(crudo.columns) == len(NOMBRES)
crudo.columns = NOMBRES

print(crudo.shape)
crudo.head(3)

## 3. Diagnóstico inicial

Porcentaje de nulos por columna. Las columnas `ip`, `telefono` y `correo` vienen vacías por la anonimización.

In [ ]:
(crudo.isna().mean() * 100).round(1).sort_values(ascending=False)

## 4. Quitar columnas que no sirven

`recopilador` dice "2025" en todas las filas, incluso en respuestas de 2026. El año se calcula desde la fecha.

In [ ]:
df = crudo.drop(columns=["n", "ip", "telefono", "correo", "recopilador"]).copy()
df.columns.tolist()

## 5. Fechas

Vienen como texto en español (`sep. 23, 2026 09:42:24 a. m.`) con un espacio especial. `fecha_es` las convierte.

In [ ]:
df["inicio"] = df["inicio"].map(fecha_es)
df["fin"] = df["fin"].map(fecha_es)
df["fecha"] = df["inicio"].dt.normalize()
df["anio"] = df["inicio"].dt.year
df["duracion_seg"] = (df["fin"] - df["inicio"]).dt.total_seconds()

print("fechas no leídas:", df["inicio"].isna().sum())
print(df["anio"].value_counts().sort_index())
print(df["duracion_seg"].describe().round(1))

## 6. Notas de satisfacción

Escala de 1 a 5. `nota_espacio` es la plataforma de conexión.

In [ ]:
NOTAS = ["nota_espacio", "nota_convocatoria", "nota_conferencistas", "nota_temas"]

df[NOTAS] = df[NOTAS].apply(pd.to_numeric, errors="coerce")
fuera_de_rango = ~df[NOTAS].isna() & ~df[NOTAS].apply(lambda s: s.between(1, 5))
print("notas fuera de 1-5:", int(fuera_de_rango.sum().sum()))
print("notas vacías:", df[NOTAS].isna().sum().to_dict())

df["promedio_satisfaccion"] = df[NOTAS].mean(axis=1).round(2)
df[NOTAS + ["promedio_satisfaccion"]].describe().round(2)

## 7. Identificadores

La empresa debe ser `EMP-0000` y la persona `P-00000`. Las respuestas sin empresa se conservan, pero se marcan.

In [ ]:
formato_empresa_malo = df["empresa_id"].notna() & ~df["empresa_id"].astype(str).str.match(PATRON_EMPRESA)
formato_persona_malo = df["persona_id"].notna() & ~df["persona_id"].astype(str).str.match(PATRON_PERSONA)
print("empresa con formato malo:", int(formato_empresa_malo.sum()))
print("persona con formato malo:", int(formato_persona_malo.sum()))

df["sin_empresa"] = df["empresa_id"].isna()
print("sin empresa:", int(df["sin_empresa"].sum()), "| sin persona:", int(df["persona_id"].isna().sum()))

## 8. Textos: ciudad, actualizaciones y temas de interés

Ciudades con variantes (`Bogota`, `Bogotá D.C.`, `Medellib`) pasan a un solo nombre. En actualizaciones, las respuestas escritas a mano pasan a `Otro`.

In [ ]:
antes = df["ciudad"].nunique()
df["ciudad"] = df["ciudad"].map(ciudad_estandar)
print("ciudades distintas:", antes, "->", df["ciudad"].nunique())

df["acepta_actualizaciones"] = df["acepta_actualizaciones"].map(actualizaciones_estandar)

df["temas_interes"] = df["temas_interes"].astype("string").str.strip().replace("", pd.NA)

print(df["ciudad"].value_counts().head(8))
print(df["acepta_actualizaciones"].value_counts(dropna=False))

## 9. Cruce con el CRM

`empresa_en_crm` y `metodo_asignacion` ya vienen en la base. Se pasa a verdadero o falso y se comprueba contra el CRM real.

In [ ]:
df["empresa_en_crm"] = df["empresa_en_crm"].map({"Sí": True, "No": False}).astype("boolean")

crm = pd.read_excel(ORIGINALES / "Base de datos Hackaton_anonimizada.xlsx", usecols=["Nombre de Empresa [anonimizado]"])
codigos_crm = set(crm["Nombre de Empresa [anonimizado]"].astype(str))
esta_en_crm = df["empresa_id"].isin(codigos_crm)

print("dice Sí pero no está en el CRM:", int((df["empresa_en_crm"].fillna(False) & ~esta_en_crm).sum()))
print("dice No pero sí está en el CRM:", int(((df["empresa_en_crm"] == False) & esta_en_crm).sum()))
print(df["metodo_asignacion"].value_counts(dropna=False))

## 10. Duplicados

In [ ]:
print("respuesta_id repetido:", int(df["respuesta_id"].duplicated().sum()))
repetidas = df[df["persona_id"].notna() & df.duplicated(["persona_id", "inicio"], keep=False)]
print("misma persona y mismo instante:", len(repetidas))
repetidas[["respuesta_id", "persona_id", "inicio"]]

## 11. Columnas finales y guardado

In [ ]:
df["tipo_evento"] = "virtual"

COLUMNAS = ["respuesta_id", "tipo_evento", "inicio", "fecha", "anio", "duracion_seg", "persona_id", "empresa_id", "sin_empresa",
            "empresa_en_crm", "metodo_asignacion", "ciudad", "nota_espacio", "nota_convocatoria", "nota_conferencistas",
            "nota_temas", "promedio_satisfaccion", "acepta_actualizaciones", "temas_interes"]
limpio = df[COLUMNAS]

LIMPIOS.mkdir(parents=True, exist_ok=True)
limpio.to_csv(LIMPIOS / "encuestas_virtuales.csv", index=False, encoding="utf-8-sig")
print(limpio.shape)
limpio.head()

## 12. Resumen de calidad

In [ ]:
resumen = {
    "filas": len(limpio),
    "respuestas 2025": int((limpio["anio"] == 2025).sum()),
    "respuestas 2026": int((limpio["anio"] == 2026).sum()),
    "sin empresa": int(limpio["sin_empresa"].sum()),
    "empresa en el CRM": int((limpio["empresa_en_crm"] == True).sum()),
    "empresa fuera del CRM": int((limpio["empresa_en_crm"] == False).sum()),
    "promedio de satisfacción 2026": round(limpio.loc[limpio["anio"] == 2026, "promedio_satisfaccion"].mean(), 2),
}
pd.Series(resumen)